In [ ]:
from pathlib import Path

import pandas as pd


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/processed/idealistaAPI")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"
SALE_PATH = DATA_DIR / "sale_homes_clean.csv"
RENT_PATH = DATA_DIR / "rent_homes_clean.csv"
OUTPUT_PATH = DATA_DIR / "real_estate_dataset_ml.csv"

required_columns = [
    "codigo_inmueble",
    "precio",
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "tipologia",
    "subtipologia",
    "provincia",
    "municipio",
    "distrito",
    "latitud",
    "longitud",
    "planta",
    "es_exterior",
    "tiene_ascensor",
    "tiene_garaje",
    "obra_nueva",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

final_columns = [
    "tipo_operacion",
    "precio",
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "tipologia",
    "subtipologia",
    "provincia",
    "municipio",
    "distrito",
    "latitud",
    "longitud",
    "planta",
    "es_exterior",
    "tiene_ascensor",
    "tiene_garaje",
    "obra_nueva",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

text_cols = ["tipologia", "subtipologia", "provincia", "municipio", "distrito", "planta"]
bool_cols = ["es_exterior", "tiene_ascensor", "tiene_garaje", "obra_nueva"]
numeric_cols = [
    "precio",
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "latitud",
    "longitud",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]


def clean_dataset(df: pd.DataFrame, tipo_operacion: str) -> pd.DataFrame:
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Faltan columnas en {tipo_operacion}: {missing_columns}")

    out = df[required_columns].copy()
    out["tipo_operacion"] = tipo_operacion

    for col in text_cols + bool_cols:
        out[col] = out[col].replace(r"^\s*$", pd.NA, regex=True)

    for col in text_cols:
        out[col] = out[col].fillna("desconocido").astype("string")

    bool_map = {
        True: True,
        False: False,
        "True": True,
        "False": False,
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }
    for col in bool_cols:
        out[col] = out[col].map(lambda x: bool_map.get(x, pd.NA)).astype("boolean").fillna(False)

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=["precio"])
    for col in [c for c in numeric_cols if c != "precio"]:
        median_value = out[col].median()
        if pd.notna(median_value):
            out[col] = out[col].fillna(median_value)

    return out


for path in [SALE_PATH, RENT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo: {path}")

df_venta = clean_dataset(pd.read_csv(SALE_PATH), "venta")
df_alquiler = clean_dataset(pd.read_csv(RENT_PATH), "alquiler")

df_total = pd.concat([df_venta, df_alquiler], ignore_index=True)
rows_before = len(df_total)
df_total = df_total.drop_duplicates(subset=["codigo_inmueble", "tipo_operacion"]).copy()
rows_after = len(df_total)

df_total = df_total[final_columns]
df_total.to_csv(OUTPUT_PATH, index=False)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Venta: {df_venta.shape} | Alquiler: {df_alquiler.shape} | Total final: {df_total.shape}")
print(f"Duplicados eliminados: {rows_before - rows_after}")
print(f"CSV unificado guardado en: {OUTPUT_PATH}")
print("Nulos restantes por columna:")
print(df_total.isna().sum().sort_values(ascending=False).to_string())